# 02 — Data Quality and Flood Events

Two jobs, and they are related.

**First: how good is the data, sensor by sensor, year by year?** Not "is it
good" — *where* and *when* is it good, because that turns out to change a lot
across seven years, and it changes most in the years we will be testing on.

**Second: what is a flood?** Every metric in this project, every alert threshold
and every claim about accuracy rests on this definition. It gets derived from
evidence here, and it gets stress-tested rather than asserted.

**Runtime:** about 3 minutes on the cleaned Parquet from notebook 01.

**Outputs, written to `docs/reports/phase0/`:**

| File | Contents |
|---|---|
| `quality_scorecard.csv` | one row per sensor per year: completeness, nulls, offline flag |
| `quality_null_trend.csv` | missing-value share by dataset and year |
| `class_balance.csv` | how rare a flood is, per year, per tier |
| `flood_excursions_{tier}cm.csv` | every raw run above each tier (the scan cache) |
| `flood_events.csv` | the events, after the persistence / merge / minimum rules |
| `event_definition_sensitivity.csv` | what happens to the count if you change the rules |
| `flood_hotspots.csv` | the sites that flood most |

## Setup

In [1]:
import os, sys, time, json
from pathlib import Path

_here = Path.cwd()
_root = next(p for p in [_here, *_here.parents] if (p / "config/config.yaml").is_file())
sys.path.insert(0, str(_root / "src"))
os.chdir(_root)

import numpy as np
import pandas as pd
pd.set_option("display.width", 175)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)

from bkkflood.config import load_config
from bkkflood.rawio import DATASETS, connect, interim_sql
from bkkflood.quality import quality_scorecard, station_year_profile
from bkkflood.events import detect_excursions, assemble_events, class_balance

CFG = load_config()
EV = CFG["flood_event"]
REPORTS = Path(CFG["paths"]["reports"]) / "phase0"
REPORTS.mkdir(parents=True, exist_ok=True)
con = connect()

print("tiers (cm)            :", EV["tiers_cm"])
print("primary tier          :", EV["primary_tier_cm"], "cm")
print("persistence readings  :", EV["persistence_readings"],
      f"(= {EV['persistence_readings'] * CFG['data']['cadence_minutes']} minutes)")
print("merge gap             :", EV["merge_gap_minutes"], "minutes")
print("minimum event length  :", EV["min_event_minutes"], "minutes")

tiers (cm)            : {'nuisance': 5, 'advisory': 15, 'severe': 30}
primary tier          : 15 cm
persistence readings  : 2 (= 10 minutes)
merge gap             : 60 minutes
minimum event length  : 10 minutes


---
# Part 1 — Data quality

## 1.1 The scorecard

One row per sensor per year. `null_pct` is the share of readings where the
sensor's primary measurement is missing — the road-depth reading for flood
sensors, the 1-hour accumulation for rain gauges, `wl_in` for canal levels,
discharge for flow meters.

Notebook 00 showed that the *rows* are essentially all present. What is missing
lives inside them.

In [2]:
t0 = time.time()
sc = quality_scorecard(con=con)
sc.to_csv(REPORTS / "quality_scorecard.csv", index=False)
print(f"{len(sc):,} station-years scored in {time.time() - t0:.0f}s")
sc.head(3)

3,736 station-years scored in 13s


,dataset,year,station_code,station_name,rows,expected_rows,row_completeness_pct,n_timestamps,duplicate_timestamps,ts_min,ts_max,primary_column,null_pct,sensor_effectively_offline
0,flood,2019,FL.BBN.01,ถ.เอกชัย ช่วงห้างบิ๊กซีบางบอน,105120,105120,100.0,105120,0,2019-01-01,2019-12-31 23:55:00,flood,0.0,False
1,flood,2019,FL.BBN.02,ถ.เอกชัย ซอย 60,105120,105120,100.0,105120,0,2019-01-01,2019-12-31 23:55:00,flood,0.0,False
2,flood,2019,FL.BKA.01,ถ.พหลโยธิน ช่วงวงเวียนบางเขน,105120,105120,100.0,105120,0,2019-01-01,2019-12-31 23:55:00,flood,0.0,False


## 1.2 The finding: the sensor network is getting worse

In [3]:
trend = (sc.groupby(["dataset", "year"])
           .apply(lambda g: pd.Series({
               "stations": g.station_code.nunique(),
               "null_pct": round(100 * (g["rows"] * g["null_pct"] / 100).sum() / g["rows"].sum(), 2),
               "stations_over_50pct_null": int(g.sensor_effectively_offline.sum()),
           }), include_groups=False)
           .reset_index())
trend.to_csv(REPORTS / "quality_null_trend.csv", index=False)

print("Share of readings that are MISSING, by dataset and year (%)")
display(trend.pivot(index="year", columns="dataset", values="null_pct"))

print()
print("Sensors more than half offline for the whole year")
display(trend.pivot(index="year", columns="dataset", values="stations_over_50pct_null"))

Share of readings that are MISSING, by dataset and year (%)


dataset,flood,flow,rain,water
year,,,,
2019,0.00,5.04,0.00,4.86
2020,0.03,3.44,0.08,0.86
2021,1.89,6.63,1.05,1.77
2022,2.92,5.55,2.41,4.90
2023,10.11,7.92,2.51,5.24
2024,8.20,11.22,1.97,11.65
2025,10.70,15.93,3.43,9.02



Sensors more than half offline for the whole year


dataset,flood,flow,rain,water
year,,,,
2019,0.0,0.0,0.0,2.0
2020,0.0,0.0,0.0,1.0
2021,3.0,2.0,1.0,4.0
2022,2.0,1.0,3.0,13.0
2023,12.0,1.0,3.0,13.0
2024,7.0,3.0,2.0,44.0
2025,11.0,5.0,5.0,26.0


> **This is the most important quality finding in Phase 0, and no earlier document
> in this project mentions it.**
>
> Missing readings roughly *triple* between 2022 and 2025 — flood 2.9% to 10.7%,
> flow 5.6% to 15.9%, water 4.9% to 9.0%. The network is growing in station count
> and degrading in reliability at the same time.
>
> Two consequences, and both change how the rest of the project must be built:
>
> **1. The test years are the worst years.** In a chronological split, 2023–2025
> are the test years. The model is being graded on the least reliable data in the
> archive. That is the right thing to do — it is what the future looks like — but
> it means an apparent drop in accuracy over time may be a data story, not a model
> story, and the evaluation must be able to tell the two apart (spec §E.8 adds a
> data-quality stratification for exactly this).
>
> **2. The model must be told when a sensor is offline.** A missing reading is not
> a dry road. If the model cannot distinguish them it will learn that certain
> stations "stopped flooding" in 2023, which is false and dangerous. Hence the
> `*_offline_share` features in the spec's feature list — they exist because of
> this table.

## 1.3 The worst offenders

In [4]:
worst = (sc[sc.null_pct > 50]
         .sort_values(["dataset", "null_pct"], ascending=[True, False])
         [["dataset", "year", "station_code", "station_name", "rows", "null_pct"]])
print(f"{len(worst)} station-years are more than half missing")
display(worst.groupby("dataset").size().rename("station_years"))
print()
display(worst.head(15))

164 station-years are more than half missing


dataset
flood     35
flow      12
rain      14
water    103
Name: station_years, dtype: int64

,dataset,year,station_code,station_name,rows,null_pct
305,flood,2022,FL.BKL.01,ถ.เจริญกรุง ช่วงวัดราชสิงขร,105120,100.0
351,flood,2022,FL.LSI.01,ซ.แจ้งวัฒนะ 14,105120,100.0
408,flood,2023,FL.BKL.01,ถ.เจริญกรุง ช่วงวัดราชสิงขร,105120,100.0
440,flood,2023,FL.DST.02,ถ.สามเสน ช่วงแยกเกียกกาย,105120,100.0
455,flood,2023,FL.LSI.01,ซ.แจ้งวัฒนะ 14,105120,100.0
457,flood,2023,FL.LSI.03,ถ.แจ้งวัฒนะ หน้าศาลปกครองกลาง,105120,100.0
469,flood,2023,FL.PTW.04,ถ.พระรามที่ 4 หน้าวัน แบงค็อก,105120,100.0
511,flood,2024,FL.BKA.01,ถ.พหลโยธิน ช่วงวงเวียนบางเขน,105408,100.0
515,flood,2024,FL.BKL.01,ถ.เจริญกรุง ช่วงวัดราชสิงขร,105408,100.0
546,flood,2024,FL.DST.01,ถ.พิษณุโลก ช่วงบ้านพิษณุโลก,105408,100.0


In [5]:
# Sensors that were installed part-way through a year are a different problem
# from sensors that broke. Separate them by looking at when they first reported.
first_seen = sc.groupby(["dataset", "station_code"])["ts_min"].min()
late = sc.merge(first_seen.rename("first_ever").reset_index(),
                on=["dataset", "station_code"])
late["started_mid_year"] = (
    pd.to_datetime(late.first_ever).dt.year == late.year
) & (pd.to_datetime(late.first_ever).dt.dayofyear > 7)
print("Station-years where the sensor was INSTALLED mid-year (not a fault):")
display(late[late.started_mid_year][["dataset", "year", "station_code", "first_ever"]]
        .sort_values(["dataset", "year"]).head(20))

Station-years where the sensor was INSTALLED mid-year (not a fault):


,dataset,year,station_code,first_ever
240,flood,2021,FL.DST.08,2021-08-11
255,flood,2021,FL.MBR.01,2021-08-14


## 1.4 Value ranges — what each sensor actually reports

In [6]:
ranges = []
for ds in DATASETS:
    prof = station_year_profile(ds, con=con)
    for col in CFG["data"]["schema"][ds]["values"]:
        ranges.append({
            "dataset": ds, "column": col,
            "min": prof[f"min_{col}"].min(),
            "max": prof[f"max_{col}"].max(),
            "null_pct": round(100 * (1 - prof[f"n_{col}"].sum() / prof["rows"].sum()), 2),
        })
    globals()[f"prof_{ds}"] = prof
rng = pd.DataFrame(ranges)
rng.to_csv(REPORTS / "quality_value_ranges.csv", index=False)
rng

,dataset,column,min,max,null_pct
0,flood,flood,0.00,148.80,4.98
1,flow,flow,-3297.70,3801.81,7.96
2,flow,wl,-3.42,4.45,7.99
3,flow,area,0.00,3268.33,9.24
4,flow,mean_velocity,-9.98,10.00,7.91
5,rain,rf5min,0.00,30.00,1.64
6,rain,rf15min,0.00,60.00,1.64
7,rain,rf30min,0.00,93.00,1.64
8,rain,rf1hr,0.00,124.00,1.64
9,rain,rf3hr,0.00,193.00,1.64


Three things in that table need saying out loud.

**`wl_out02` is 99.6% missing and `wl_out01` is 82% missing.** `wl_out02` is not
data, it is a column header. It is excluded in `config.yaml`. `wl_out01` is
usable but sparse and must always be treated as optional.

**Canal water level runs from about −5 m to +4 m, and we do not know what zero
is.** No datum was supplied — mean sea level, canal bed, or a local benchmark. So
only *changes* in level can be used as model inputs. "This canal is 30 cm below
its bank" is a far stronger predictor than "this canal rose 8 cm", and we cannot
compute the first one. That single missing fact is why canal level is a weak
feature in this project.

**`rf5min` never exceeds 30.0 mm anywhere, ever.** Whether that is a physical
fact or an instrument limit is worth two minutes of checking rather than a
guess — see below.

In [7]:
mx = prof_rain.groupby("station_code")["max_rf5min"].max()
print("Per-station maximum 5-minute rainfall (mm), across all seven years:")
print(mx.describe().round(2).to_string())
print()
n_at_cap = int((mx >= 29.99).sum())
print(f"gauges whose all-time maximum is exactly 30.0 mm : {n_at_cap} of {len(mx)}")
print(f"gauges that ever exceed 30.0 mm                  : {int((mx > 30.0).sum())}")

Per-station maximum 5-minute rainfall (mm), across all seven years:
count    131.00
mean      16.79
std        3.88
min       12.00
25%       14.25
50%       15.50
75%       18.50
max       30.00

gauges whose all-time maximum is exactly 30.0 mm : 1 of 131
gauges that ever exceed 30.0 mm                  : 0


> **The hypothesis was wrong, and that is worth recording.**
>
> A whole-archive maximum of exactly 30.0 mm looked like a device ceiling — many
> independent instruments stopping at the same round number usually means a field
> limit, not a coincidence. If true, it would mean 5-minute rainfall is censored
> exactly where a flash-flood model needs it most.
>
> It is not true. **One gauge of 131 reaches 30.0 mm, once.** The rest top out
> between 12 and about 25 mm, which is a smooth, believable distribution of
> station maxima. There is no shared cap; there is one gauge that saw a genuinely
> extreme five minutes.
>
> Two lessons worth keeping. First, "the maximum is a round number" is a hint, not
> a finding — the distribution below it is what settles the question. Second, this
> check nearly destroyed its own evidence: the range check in `config.yaml` was
> briefly set to null anything above 25 mm, which silently deleted the readings
> needed to answer it. A data check that sits close to real values stops being a
> check and becomes a hidden edit. The ceiling is now 60 mm — beyond any observed
> rainfall on Earth over five minutes, so it removes only nonsense.

In [8]:
# The flow meters that are not canal meters.
fw = (prof_flow.groupby("station_code")
      .agg(min_flow=("min_flow", "min"), max_flow=("max_flow", "max"))
      .sort_values("max_flow", ascending=False))
print("Flow stations by peak discharge (m3/s):")
display(fw.head(5))
print("...")
display(fw.tail(3))
print()
print("configured exclusions from canal aggregates :",
      CFG["exclusions"]["flow_stations_from_canal_aggregate"])
print("configured dead sensors                     :",
      CFG["exclusions"]["dead_sensors"])
print()
dead = fw[(fw.max_flow == 0) & (fw.min_flow == 0)]
print("stations that are flat zero for seven years  :", list(dead.index))

Flow stations by peak discharge (m3/s):


,min_flow,max_flow
station_code,,
FW.PKG.01,-3297.70,3801.81
FW.LPW.01,-16.14,2916.71
FW.BKG.01,-62.04,1000.00
FW.PNJ.01,0.00,591.25
FW.KKY.01,0.00,327.54


...


,min_flow,max_flow
station_code,,
FW.SBR.01,-13.86,10.63
FW.SSB.03,-28.16,1.45
FW.SSM.01,0.00,0.00



configured exclusions from canal aggregates : ['FW.PKG.01', 'FW.LPW.01']
configured dead sensors                     : ['FW.SSM.01']

stations that are flat zero for seven years  : ['FW.SSM.01']


> `FW.PKG.01` peaks near 3,800 m³/s and `FW.LPW.01` near 2,900. The median canal
> meter peaks below 100. These two are **river-scale gauges on the Chao Phraya**,
> and they are not faulty — an earlier version of this project called `FW.PKG.01`
> a broken sensor, which would have been an embarrassing thing to tell BMA, who
> know perfectly well that their river gauge works.
>
> They are excluded from *canal averages* because averaging a river with a canal
> destroys the canal signal, not because the data is wrong. They should be
> surfaced separately as river discharge.
>
> `FW.SSM.01` really is dead: exactly 0.00 for every reading in every year.
>
> That leaves **27 usable canal flow meters** out of 30.

---
# Part 2 — What is a flood?

## 2.1 Why `depth > 0` cannot be the answer

In [9]:
cb = class_balance(con=con)
cb.to_csv(REPORTS / "class_balance.csv", index=False)

tot = cb.sum(numeric_only=True)
rows = int(tot["rows"])
print(f"{'condition':<28}{'rows':>14}{'share':>12}{'odds':>16}")
print("-" * 70)
for label, n in [("flood is missing", int(tot["rows_null"])),
                 ("flood == 0 exactly", int(tot["rows_zero"])),
                 ("flood > 0", int(tot["n_gt0"])),
                 ("flood >= 5 cm", int(tot["n_ge5"])),
                 ("flood >= 15 cm", int(tot["n_ge15"])),
                 ("flood >= 30 cm", int(tot["n_ge30"]))]:
    print(f"{label:<28}{n:>14,}{100 * n / rows:>11.4f}%{'1 in ' + format(round(rows / n), ',') :>16}")
print("-" * 70)
print(f"{'TOTAL':<28}{rows:>14,}")
print()
print(f"deepest reading ever recorded: {cb.max_depth_cm.max()} cm")

condition                             rows       share            odds
----------------------------------------------------------------------
flood is missing                 3,782,275     4.9811%         1 in 20
flood == 0 exactly              71,492,547    94.1529%          1 in 1
flood > 0                          657,530     0.8659%        1 in 115
flood >= 5 cm                       50,929     0.0671%      1 in 1,491
flood >= 15 cm                      15,024     0.0198%      1 in 5,054
flood >= 30 cm                       3,262     0.0043%     1 in 23,278
----------------------------------------------------------------------
TOTAL                           75,932,352

deepest reading ever recorded: 148.8 cm


> **0.87% of readings are non-zero, but only 0.067% reach 5 cm.** The 13-fold gap
> between them is the noise floor: a device sitting on a wet road reporting a few
> millimetres. If `depth > 0` were the definition of a flood, Bangkok would flood
> 657,530 times in seven years, and the model would spend all its capacity
> learning to predict damp tarmac.
>
> A threshold has to sit above that floor. The cell below shows where the floor
> is.

In [10]:
q = f'''
    SELECT CASE
             WHEN flood = 0             THEN '0.0 exactly'
             WHEN flood < 0.5           THEN 'under 0.5 cm'
             WHEN flood < 1             THEN '0.5 - 1 cm'
             WHEN flood < 2             THEN '1 - 2 cm'
             WHEN flood < 5             THEN '2 - 5 cm'
             WHEN flood < 15            THEN '5 - 15 cm'
             WHEN flood < 30            THEN '15 - 30 cm'
             ELSE '30 cm and above'
           END AS band,
           count(*)::BIGINT AS rows
    FROM {interim_sql("flood")}
    WHERE flood IS NOT NULL
    GROUP BY 1
'''
order = ['0.0 exactly', 'under 0.5 cm', '0.5 - 1 cm', '1 - 2 cm', '2 - 5 cm',
         '5 - 15 cm', '15 - 30 cm', '30 cm and above']
bands = con.execute(q).fetchdf().set_index("band").reindex(order).fillna(0)
bands["share_pct"] = (100 * bands["rows"] / bands["rows"].sum()).round(4)
bands["of_nonzero_pct"] = (100 * bands["rows"] / bands["rows"][1:].sum()).round(2)
bands.loc["0.0 exactly", "of_nonzero_pct"] = np.nan
display(bands)

noise = bands.loc[["under 0.5 cm", "0.5 - 1 cm", "1 - 2 cm"], "rows"].sum()
nonzero = bands["rows"][1:].sum()
print(f"\nOf every non-zero reading, {100 * noise / nonzero:.1f}% is under 2 cm.")

,rows,share_pct,of_nonzero_pct
band,,,
0.0 exactly,71492547,99.0887,NaN
under 0.5 cm,429556,0.5954,65.33
0.5 - 1 cm,64359,0.0892,9.79
1 - 2 cm,41506,0.0575,6.31
2 - 5 cm,71180,0.0987,10.83
5 - 15 cm,35905,0.0498,5.46
15 - 30 cm,11762,0.0163,1.79
30 cm and above,3262,0.0045,0.50



Of every non-zero reading, 81.4% is under 2 cm.


## 2.2 The tiers, and what they mean operationally

In [11]:
pd.DataFrame([
    (5,  "nuisance", "Water on the road. Visible, annoying, not dangerous.", "Watch"),
    (15, "advisory", "Traffic disrupted; small vehicles struggle. ALL HEADLINE METRICS ARE REPORTED HERE.", "Advisory"),
    (30, "severe",   "Impassable; possible road closure.", "Warning"),
], columns=["tier_cm", "name", "what it means on the ground", "alert level"])

,tier_cm,name,what it means on the ground,alert level
0,5,nuisance,"Water on the road. Visible, annoying, not dang...",Watch
1,15,advisory,Traffic disrupted; small vehicles struggle. AL...,Advisory
2,30,severe,Impassable; possible road closure.,Warning


These match the thresholds in the supervisor's dashboard scheme exactly
(&lt;5 / 5–15 / 15–30 / &gt;30 cm), which is a useful coincidence: our modelling tiers
and the operational display speak the same language.

## 2.3 The scan

`detect_excursions` finds every unbroken run of readings above a tier. No rules
applied yet — a single 5-minute blip shows up here as a one-reading excursion.
This is the expensive step, so its output is cached to CSV; the rules are applied
afterwards in memory, which is what makes the sensitivity analysis below
affordable.

In [12]:
TIERS = sorted(EV["tiers_cm"].values())
excursions = {}
for tier in TIERS:
    cache = REPORTS / f"flood_excursions_{tier}cm.csv"
    if cache.exists():
        exc = pd.read_csv(cache, parse_dates=["started_at", "ended_at"])
        print(f"  tier {tier:>2} cm : {len(exc):>6,} excursions  (from cache)")
    else:
        t0 = time.time()
        exc = detect_excursions(tier, con=con)
        exc.to_csv(cache, index=False)
        print(f"  tier {tier:>2} cm : {len(exc):>6,} excursions  ({time.time() - t0:.0f}s)")
    excursions[tier] = exc

  tier  5 cm :  5,281 excursions  (from cache)
  tier 15 cm :  1,469 excursions  (from cache)
  tier 30 cm :    342 excursions  (from cache)


## 2.4 Why two consecutive readings, and not one or three

In [13]:
tier = EV["primary_tier_cm"]
exc = excursions[tier]
dist = (exc.n_readings.clip(upper=12).value_counts().sort_index()
        .rename("excursions").to_frame())
dist.index.name = "consecutive readings above the tier"
dist["cumulative_kept"] = dist.excursions[::-1].cumsum()[::-1]
dist["minutes"] = dist.index * CFG["data"]["cadence_minutes"]
display(dist)

one = int((exc.n_readings == 1).sum())
print(f"\nAt the {tier} cm tier, {one:,} of {len(exc):,} excursions "
      f"({100 * one / len(exc):.0f}%) are a SINGLE 5-minute reading.")

,excursions,cumulative_kept,minutes
consecutive readings above the tier,,,
1,470,1469,5
2,158,999,10
3,101,841,15
4,84,740,20
5,59,656,25
6,54,597,30
7,55,543,35
8,48,488,40
9,32,440,45



At the 15 cm tier, 470 of 1,469 excursions (32%) are a SINGLE 5-minute reading.


> A third of all excursions are one reading long. A single 5-minute spike above
> 15 cm with dry readings either side is a splash, a passing truck, or a corrupted
> packet — not a flood. Requiring two consecutive readings removes them.
>
> Requiring three or more would start discarding real events: Bangkok's floods are
> short (see 2.6), and a 15-minute flash flood is still a flash flood. Two is
> where the curve elbows, and the table above is why — the drop from 1 to 2 is
> large, and every step after that is small.

## 2.5 The rules, applied — and what happens if you choose differently

In [14]:
events = pd.concat([assemble_events(excursions[t]) for t in TIERS], ignore_index=True)
events.to_csv(REPORTS / "flood_events.csv", index=False)

summary = (events.groupby("tier_cm")
           .agg(events=("station_code", "size"),
                stations_affected=("station_code", "nunique"),
                median_minutes=("duration_minutes", "median"),
                p90_minutes=("duration_minutes", lambda s: s.quantile(0.90)),
                longest_minutes=("duration_minutes", "max"),
                deepest_cm=("peak_depth_cm", "max"))
           .reset_index())
display(summary)

print()
print(f"At the primary {tier} cm tier there are "
      f"{int(summary.loc[summary.tier_cm == tier, 'events'].iloc[0]):,} flood events "
      f"in the ENTIRE seven-year archive.")

,tier_cm,events,stations_affected,median_minutes,p90_minutes,longest_minutes,deepest_cm
0,5.0,3135,104,45.0,170.0,3995,148.8
1,15.0,837,83,45.0,185.0,2565,148.8
2,30.0,132,38,65.0,219.0,2320,148.8



At the primary 15 cm tier there are 837 flood events in the ENTIRE seven-year archive.


> ### The number that decides the modelling strategy
>
> **837 flood events at 15 cm. Not per year — in total, across all of Bangkok,
> across seven years.** At 30 cm there are 132.
>
> That is why the spec chooses gradient-boosted trees with onset specialists over
> a Transformer or a Temporal Fusion Transformer. A TFT has tens of thousands of
> parameters and needs thousands of events; given 837 it will memorise them.
> This is not a preference about model families, it is arithmetic.
>
> **Correction to spec §B.4.** The spec quotes **999** events at 15 cm. That figure
> is the count of *excursions surviving the persistence rule* — it was computed
> before the merge-gap and minimum-duration rules were applied. Under the full
> definition the number is **837**. Both come from the same scan; the chain is
> 1,469 raw excursions → 999 after persistence → 837 after merging. The argument
> the spec makes from that number is unaffected, but the number itself is now
> correct and reproducible from this notebook.

In [15]:
# How sensitive is the count to the three choices? A definition nobody has
# stress-tested is a definition nobody should trust.
sens = []
for tier in TIERS:
    e = excursions[tier]
    for persistence in [1, 2, 3, 4, 6]:
        for gap in [0, 15, 30, 60, 120, 180]:
            ev = assemble_events(e, persistence_readings=persistence,
                                 merge_gap_minutes=gap)
            sens.append({"tier_cm": tier, "persistence_readings": persistence,
                         "merge_gap_minutes": gap, "events": len(ev),
                         "median_minutes": ev.duration_minutes.median() if len(ev) else np.nan})
sens = pd.DataFrame(sens)
sens.to_csv(REPORTS / "event_definition_sensitivity.csv", index=False)

print(f"Event count at the {EV['primary_tier_cm']} cm tier under different rules")
print("(rows = consecutive readings required, columns = merge gap in minutes)")
display(sens[sens.tier_cm == EV["primary_tier_cm"]]
        .pivot(index="persistence_readings", columns="merge_gap_minutes", values="events"))

Event count at the 15 cm tier under different rules
(rows = consecutive readings required, columns = merge gap in minutes)


merge_gap_minutes,0,15,30,60,120,180
persistence_readings,,,,,,
1,999,898,867,859,853,848
2,999,885,851,837,827,823
3,841,782,765,754,745,742
4,740,706,694,684,674,672
6,597,579,573,569,563,560


Read that table the way you would read a stress test.

**Down the rows (persistence).** Going from 1 to 2 readings is a big cut — that is
the noise coming out. From 2 to 3 to 4 the losses are steady and smaller: those
are real short floods being discarded. The elbow is at 2.

**Across the columns (merge gap).** From 0 to 60 minutes the count falls steeply
as one flooded afternoon stops being counted as a dozen separate events. Past 60
it flattens, and beyond that we would start merging genuinely separate storms into
one.

The configured choice — 2 readings, 60 minutes — sits at the elbow of both curves.
That is the justification, and it is measured rather than asserted. If a reviewer
prefers different values, this table tells them exactly what it would cost.

## 2.6 What a Bangkok flood actually looks like

In [16]:
for tier in TIERS:
    e = events[events.tier_cm == tier]
    q = e.duration_minutes.quantile([0.25, 0.5, 0.75, 0.9, 0.99]).round(0)
    print(f"tier {tier:>2} cm  n={len(e):>5,}   duration minutes: "
          f"p25={q[0.25]:>5.0f}  median={q[0.5]:>5.0f}  p75={q[0.75]:>6.0f}  "
          f"p90={q[0.9]:>6.0f}  p99={q[0.99]:>7.0f}")
print()
e = events[events.tier_cm == EV["primary_tier_cm"]]
print(f"At {EV['primary_tier_cm']} cm, "
      f"{100 * (e.duration_minutes <= 60).mean():.0f}% of events last an hour or less "
      f"and {100 * (e.duration_minutes <= 180).mean():.0f}% last three hours or less.")

tier  5 cm  n=3,135   duration minutes: p25=   20  median=   45  p75=    90  p90=   170  p99=    613
tier 15 cm  n=  837   duration minutes: p25=   25  median=   45  p75=    95  p90=   185  p99=    608
tier 30 cm  n=  132   duration minutes: p25=   24  median=   65  p75=   130  p90=   219  p99=    951

At 15 cm, 61% of events last an hour or less and 90% last three hours or less.


> **These are flash events, not river floods.** Most are over within an hour.
>
> That single fact explains the hardest problem in this project. A 6-hour forecast
> of something that begins and ends inside 45 minutes is a very different task
> from forecasting a river crest that builds over two days — and it is why lead
> time, not recall, is the honest measure of whether this system warns anyone
> (spec §E.8).

## 2.7 Events by year — and why a single accuracy number would mislead

In [17]:
by_year = (events.groupby(["tier_cm", "year"]).size()
           .rename("events").reset_index()
           .pivot(index="year", columns="tier_cm", values="events").fillna(0).astype(int))
display(by_year)

stations = (events.groupby(["tier_cm", "year"])["station_code"].nunique()
            .reset_index().pivot(index="year", columns="tier_cm", values="station_code"))
print("Distinct stations affected")
display(stations)

p = EV["primary_tier_cm"]
print(f"At {p} cm: worst year {by_year[p].idxmax()} with {by_year[p].max()} events; "
      f"quietest year {by_year[p].idxmin()} with {by_year[p].min()}. "
      f"A {by_year[p].max() / by_year[p].min():.0f}x difference.")

tier_cm,5.0,15.0,30.0
year,,,
2019,367,97,27
2020,432,110,17
2021,593,138,15
2022,774,235,30
2023,384,99,17
2024,253,46,7
2025,332,112,19


Distinct stations affected


tier_cm,5.0,15.0,30.0
year,,,
2019,65,45,17
2020,61,38,11
2021,84,50,12
2022,89,56,16
2023,93,56,14
2024,56,25,7
2025,77,49,14


At 15 cm: worst year 2022 with 235 events; quietest year 2024 with 46. A 5x difference.


> **2022 had six times more flooding than 2024.** Any performance figure averaged
> across years hides that, and an operator planning staffing needs to know that a
> bad year looks nothing like a good one.
>
> There is a sharper consequence for the evaluation. In the spec's rolling-origin
> folds, **fold 4 validates on 2024** — the quietest year in the archive. A
> validation year with almost no positive examples cannot select a sensible
> operating threshold, and in the previous version of this project exactly that
> happened: one model trained to a single decision stump because early stopping
> read the near-absence of positives correctly. Expect it, report it, do not
> paper over it.

## 2.8 Onset versus ongoing — the distinction the whole project turns on

In [18]:
q = f'''
WITH s AS (
    SELECT station_code, ts, flood,
           lag(flood, {60 // CFG["data"]["cadence_minutes"]}) OVER (
               PARTITION BY station_code ORDER BY ts) AS flood_1h_ago
    FROM {interim_sql("flood")}
    WHERE flood IS NOT NULL
)
SELECT
  sum(CASE WHEN flood >= {p} THEN 1 ELSE 0 END)::BIGINT                                   AS rows_at_tier,
  sum(CASE WHEN flood >= {p} AND flood_1h_ago >= {p} THEN 1 ELSE 0 END)::BIGINT           AS ongoing,
  sum(CASE WHEN flood >= {p} AND (flood_1h_ago < {p}) THEN 1 ELSE 0 END)::BIGINT          AS onset,
  count(*)::BIGINT                                                                        AS rows_total
FROM s
'''
o = con.execute(q).fetchdf().iloc[0]
onset, ongoing, n_rows_all = int(o.onset), int(o.ongoing), int(o.rows_total)
print(f"Rows at or above {p} cm            : {int(o.rows_at_tier):>10,}")
print(f"  of which ONGOING (flooded 1h ago): {ongoing:>10,}  ({100*ongoing/int(o.rows_at_tier):.0f}%)")
print(f"  of which ONSET   (dry 1h ago)    : {onset:>10,}  ({100*onset/int(o.rows_at_tier):.0f}%)")
print()
print(f"Base rate of an ONSET row          : 1 in {n_rows_all // max(onset,1):,}")

Rows at or above 15 cm            :     15,024
  of which ONGOING (flooded 1h ago):      7,713  (51%)
  of which ONSET   (dry 1h ago)    :      7,311  (49%)

Base rate of an ONSET row          : 1 in 9,868


> A row where the station was **already** flooded an hour ago is *ongoing*. A row
> where it was dry is *onset*. **Only onset rows require forecasting at all** — the
> ongoing ones are answered perfectly by a one-line rule: "it was flooded, it
> still is".
>
> This is the trap the previous version of this project fell into. It reported 55%
> recall at 15 cm / 1 hour. Decomposed, that was roughly 100% on already-flooded
> rows and **9% on genuine onsets** — a monitoring tool wearing a forecasting
> label. Training a separate model on dry rows only took onset recall from 9% to
> 63%.
>
> Two rules follow, and they are not negotiable:
>
> 1. **Every recall figure in this project is reported twice** — overall, and
>    onset-only. A single number describes a monitor.
> 2. **Onset models raise a Watch, never a Warning.** Look at the base rate above:
>    onset precision in the previous round was 1–2%, which is a large lift over
>    that base rate and still means most notices do not lead to flooding. Fine for
>    "check this"; unacceptable for "close the road".

## 2.9 Hotspots

In [19]:
p = EV["primary_tier_cm"]
hot = (events[events.tier_cm == p]
       .groupby("station_code")
       .agg(events=("station_code", "size"),
            total_minutes=("duration_minutes", "sum"),
            deepest_cm=("peak_depth_cm", "max"),
            first=("started_at", "min"), last=("started_at", "max"))
       .sort_values("events", ascending=False))
names = (pd.read_csv(REPORTS / "quality_scorecard.csv")
         .query("dataset == 'flood'")[["station_code", "station_name"]]
         .drop_duplicates("station_code").set_index("station_code")["station_name"])
hot["station_name"] = hot.index.map(names)
hot.to_csv(REPORTS / "flood_hotspots.csv")
display(hot.head(15))

flood_stations = sc.query("dataset == 'flood'").station_code.nunique()
never = flood_stations - hot.shape[0]
print(f"\n{never} of {flood_stations} flood sensors have NEVER recorded an event at {p} cm.")
top10 = hot.events.head(10).sum()
print(f"The 10 worst sites account for {top10} of {int(hot.events.sum())} events "
      f"({100 * top10 / hot.events.sum():.0f}%).")

,events,total_minutes,deepest_cm,first,last,station_name
station_code,,,,,,
FL.SMI.01,48,2610,51.5,2019-07-25 12:50:00,2025-11-02 21:10:00,ถ.พหลโยธิน ช่วงซอยพหลโยธิน 60/1
FL.PWT.02,38,6335,43.2,2019-05-29 07:15:00,2025-10-04 12:50:00,ถ.เฉลิมพระเกียรติ ร.9 ช่วงแยกศรีอุดม
FL.DDG.02,32,2600,34.7,2019-06-07 14:30:00,2025-11-13 02:00:00,ถ.ประชาสงเคราะห์ ตลาดห้วยขวาง
FL.SLG.03,31,3390,40.3,2019-04-23 00:20:00,2025-11-02 22:10:00,"""ถ.พัฒนาการ ช่วงธนาคารกรุงไทย """
FL.DDG.01,29,1150,29.9,2020-09-23 15:40:00,2025-11-13 02:00:00,ถ.รัชดาภิเษก ช่วงแยกเทียมร่วมมิตร
FL.BKM.01,27,2795,39.4,2019-06-30 21:50:00,2025-11-13 01:45:00,ถ.นวมินทร์ ช่วงซอย 46 (สันติอโศก)
FL.RTW.10,25,980,44.6,2019-06-07 14:40:00,2025-11-13 02:25:00,ถ.ศรีอยุธยา ช่วงโรงเรียนศรีอยุธยา
FL.STN.03,24,2165,44.2,2019-09-14 18:35:00,2025-11-13 02:45:00,ถ.สาธุประดิษฐ์ ซอย 2
FL.LSI.04,22,1375,36.1,2019-04-26 14:25:00,2025-05-10 17:50:00,ถ.แจ้งวัฒนะขาเข้า (โลตัส)



24 of 107 flood sensors have NEVER recorded an event at 15 cm.
The 10 worst sites account for 298 of 837 events (36%).


> Flooding is extremely concentrated: a handful of sites produce most of the
> events, and a fifth of the sensors have never seen one. That is genuinely useful
> for the dashboard's hotspot panel, and it is a warning for the model — "this
> place floods often" is a strong predictor, but it is climatology, not
> meteorology, and it says nothing at all about a sensor installed last year.

## 3. Summary

In [20]:
p = EV["primary_tier_cm"]
n_events = int((events.tier_cm == p).sum())
print("=" * 76)
print("PHASE 0 COMPLETE - DATA QUALITY AND FLOOD EVENTS")
print("=" * 76)
print("DATA QUALITY")
print(f"  station-years scored              : {len(sc):,}")
fl_trend = trend[trend.dataset == "flood"].set_index("year")["null_pct"]
print(f"  flood readings missing, 2019      : {fl_trend.loc[2019]:.2f}%")
print(f"  flood readings missing, 2025      : {fl_trend.loc[2025]:.2f}%   <- degrading")
print(f"  station-years over 50% missing    : {int(sc.sensor_effectively_offline.sum())}")
print(f"  usable canal flow meters          : 27 of 30")
print()
print("THE TARGET")
print(f"  flood readings total              : {rows:,}")
print(f"  readings at or above {p} cm        : {int(tot['n_ge' + str(p)]):,}  "
      f"(1 in {rows // int(tot['n_ge' + str(p)]):,})")
print(f"  deepest reading ever              : {cb.max_depth_cm.max()} cm")
print()
print("FLOOD EVENTS (2 readings, 60 min merge, 10 min minimum)")
for t in TIERS:
    e = events[events.tier_cm == t]
    print(f"  {t:>2} cm : {len(e):>5,} events   median {e.duration_minutes.median():>3.0f} min   "
          f"{e.station_code.nunique():>3} stations")
print()
print(f"  >>> {n_events} events at the primary {p} cm tier, in seven years.")
print("      That number, not a model preference, is why this project uses")
print("      gradient-boosted trees with onset specialists (spec E.6).")
print("=" * 76)
print()
print("Next: Phase 1 - notebook 03 extracts terrain features from the 1 m DTM.")
con.close()

PHASE 0 COMPLETE - DATA QUALITY AND FLOOD EVENTS
DATA QUALITY
  station-years scored              : 3,736
  flood readings missing, 2019      : 0.00%
  flood readings missing, 2025      : 10.70%   <- degrading
  station-years over 50% missing    : 164
  usable canal flow meters          : 27 of 30

THE TARGET
  flood readings total              : 75,932,352
  readings at or above 15 cm        : 15,024  (1 in 5,054)
  deepest reading ever              : 148.8 cm

FLOOD EVENTS (2 readings, 60 min merge, 10 min minimum)
   5 cm : 3,135 events   median  45 min   104 stations
  15 cm :   837 events   median  45 min    83 stations
  30 cm :   132 events   median  65 min    38 stations

  >>> 837 events at the primary 15 cm tier, in seven years.
      That number, not a model preference, is why this project uses
      gradient-boosted trees with onset specialists (spec E.6).

Next: Phase 1 - notebook 03 extracts terrain features from the 1 m DTM.
